# 09 – Predictive analytics: will an animal find a home?

**Goal (Task 4, part 3):** a **classification** task on silver data with Spark MLlib.

- **Question:** will a shelter animal **find a home** (status `reserved` or `adopted`)? → label `found_home` = 1 / 0
- **Why it's useful:** the shelter can see early which animals are at risk of staying long and promote them.
- **Steps:** prepare the dataset → split train/test (holdout 80/20 + 5-fold cross-validation) → train a model → evaluate with several metrics → write the predictions into the pile (`gold.found_home_predictions`).

In [1]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.functions import vector_to_array

from spark_pile import get_spark, read_iceberg, write_iceberg, get_catalog

spark = get_spark("09-ml")
catalog = get_catalog("silo")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-avro_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2e313021-9b14-49b7-aaaa-2e5a6c8a9c26;1.0
	confs: [default]
	found org.apache.spark#spark-avro_2.13;4.0.1 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.tukaani#xz;1.10 in central
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found software.amazon.awssdk#bundle;2.24.6 in central
	found org.wildfly.openssl#wildfly-openssl;1.1.3.Final in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-avro_2.13/4.0.1/spark-avro_2.13-4.0.1.jar ...
	[SUCCESSFUL ] org.apache.spark#spark-

## Step 1 – Prepare the dataset (from silver)

**Feature extraction:** one row per animal, built with joins:
- from `silver.animals`: species, size, sex, age, vaccinated, **days in the shelter** (computed from `arrived_at`)
- from `silver.applications`: **number of applications** for the animal (= interest)
- from `silver.shelters`: the shelter's **capacity**

We use only *how many* applications an animal got, not their status, because the status *is* the answer (that would be "data leakage").

In [2]:
animals = read_iceberg(spark, "silver.animals")
applications = read_iceberg(spark, "silver.applications")
shelters = read_iceberg(spark, "silver.shelters")

per_animal = applications.groupBy(F.col("animal_id").alias("id")).agg(F.count("*").alias("n_applications"))

dataset = (animals
           .join(per_animal, "id", "left").fillna({"n_applications": 0})
           .join(shelters.select(F.col("id").alias("shelter_id"), F.col("capacity").alias("shelter_capacity")),
                 "shelter_id")
           .withColumn("days_in_shelter", F.datediff(F.current_date(), "arrived_at"))
           .withColumn("vaccinated", F.col("is_vaccinated").cast("double"))
           .withColumn("label", F.col("found_home").cast("double"))
           .select("id", "name", "species", "size", "sex", "age_years", "days_in_shelter",
                   "n_applications", "vaccinated", "shelter_capacity", "label"))

print("Rows:", dataset.count())
dataset.groupBy("label").count().show()

Rows: 3008
+-----+-----+
|label|count|
+-----+-----+
|  1.0| 1029|
|  0.0| 1979|
+-----+-----+



In [3]:
# Is there a signal? Share of animals that found a home, by number of applications
dataset.groupBy("n_applications").agg(F.count("*").alias("animals"), F.round(F.avg("label"), 2).alias("found_home_rate")) \
       .orderBy("n_applications").show()

+--------------+-------+---------------+
|n_applications|animals|found_home_rate|
+--------------+-------+---------------+
|             0|    564|            0.0|
|             1|    977|           0.27|
|             2|    773|           0.44|
|             3|    422|           0.52|
|             4|    189|           0.77|
|             5|     55|           0.69|
|             6|     18|           0.83|
|             7|      8|           0.88|
|             8|      2|            1.0|
+--------------+-------+---------------+



## Step 2 – Encoding and normalisation (a Spark ML Pipeline)

| Step | What | Why |
|---|---|---|
| `StringIndexer` | text categories → numbers (`dog`→0, `cat`→1, …) | models need numbers |
| `OneHotEncoder` | number → 0/1 columns | so "cat=1, dog=0" isn't read as an order |
| `VectorAssembler` | numeric columns → one vector | MLlib wants one `features` vector |
| `StandardScaler` | numbers → mean 0, std 1 | age (0–15) and days (0–700) on the same scale |

In [4]:
CATEGORICAL = ["species", "size", "sex"]
NUMERIC = ["age_years", "days_in_shelter", "n_applications", "vaccinated", "shelter_capacity"]

prep_stages = (
    [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in CATEGORICAL]
    + [OneHotEncoder(inputCols=[f"{c}_idx" for c in CATEGORICAL], outputCols=[f"{c}_ohe" for c in CATEGORICAL]),
       VectorAssembler(inputCols=NUMERIC, outputCol="numeric_vec", handleInvalid="keep"),
       StandardScaler(inputCol="numeric_vec", outputCol="numeric_scaled", withMean=True, withStd=True),
       VectorAssembler(inputCols=["numeric_scaled"] + [f"{c}_ohe" for c in CATEGORICAL], outputCol="features")]
)

## Step 3 – Split into training and test set (holdout 80 / 20)

The model learns on **train** and is judged on **test**, data it has never seen. `seed=42` makes the split repeatable.

In [5]:
train, test = dataset.randomSplit([0.8, 0.2], seed=42)
print("train:", train.count(), "| test:", test.count())

train: 2415 | test: 593


## Step 4 – Train classification models

1. **Logistic regression**, tuned with **5-fold cross-validation** on the training set (tries 3 values of the regularisation parameter and keeps the best).
2. **Random forest** (100 decision trees) for comparison.

In [6]:
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

logreg = LogisticRegression(labelCol="label", featuresCol="features")
cv = CrossValidator(
    estimator=Pipeline(stages=prep_stages + [logreg]),
    estimatorParamMaps=ParamGridBuilder().addGrid(logreg.regParam, [0.0, 0.01, 0.1]).build(),
    evaluator=auc, numFolds=5, seed=42)
logreg_model = cv.fit(train)
print("5-fold CV, average AUC per regParam [0.0, 0.01, 0.1]:", [round(float(m), 3) for m in logreg_model.avgMetrics])

forest_model = Pipeline(stages=prep_stages + [
    RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=6, seed=42)]).fit(train)

5-fold CV, average AUC per regParam [0.0, 0.01, 0.1]: [0.765, 0.765, 0.765]


## Step 5 – Evaluate on the test set

- **Accuracy**: share of correct predictions
- **Precision**: of the animals predicted to find a home, how many really did
- **Recall**: of the animals that really found a home, how many the model found
- **F1**: balance of precision and recall
- **ROC-AUC**: how well the model ranks the animals (0.5 = guessing, 1.0 = perfect)
- **Baseline**: always predicting "no home" (the bigger class), the score to beat

In [7]:
def evaluate(model, name):
    pred = model.transform(test)
    m = lambda metric, **kw: MulticlassClassificationEvaluator(labelCol="label", metricName=metric, **kw).evaluate(pred)
    return {"model": name,
            "accuracy": m("accuracy"),
            "precision (found home)": m("precisionByLabel", metricLabel=1.0),
            "recall (found home)": m("recallByLabel", metricLabel=1.0),
            "F1 (found home)": m("fMeasureByLabel", metricLabel=1.0),
            "ROC-AUC": auc.evaluate(pred)}

baseline = 1 - test.agg(F.avg("label")).first()[0]
results = pd.DataFrame([evaluate(logreg_model, "logistic regression (5-fold CV)"),
                        evaluate(forest_model, "random forest"),
                        {"model": "baseline: always 'no home'", "accuracy": baseline}]).round(3)
results

,model,accuracy,precision (found home),recall (found home),F1 (found home),ROC-AUC
0,logistic regression (5-fold CV),0.734,0.688,0.293,0.410,0.750
1,random forest,0.708,0.590,0.261,0.362,0.754
2,baseline: always 'no home',0.683,NaN,NaN,NaN,NaN


In [8]:
# Confusion matrix of the logistic regression
logreg_model.transform(test).groupBy("label").pivot("prediction", [0.0, 1.0]).count() \
    .withColumnRenamed("0.0", "predicted 0").withColumnRenamed("1.0", "predicted 1").orderBy("label").show()

+-----+-----------+-----------+
|label|predicted 0|predicted 1|
+-----+-----------+-----------+
|  0.0|        380|         25|
|  1.0|        133|         55|
+-----+-----------+-----------+



In [9]:
# What did the random forest find important?
names = NUMERIC + [f"{c}={v}" for c, idx in zip(CATEGORICAL, forest_model.stages[:3]) for v in idx.labels]
importance = forest_model.stages[-1].featureImportances.toArray()
pd.Series(importance, index=names).sort_values(ascending=False).head(8).round(3)

n_applications      0.684
age_years           0.083
shelter_capacity    0.077
days_in_shelter     0.070
vaccinated          0.013
size=large          0.011
sex=male            0.010
sex=female          0.009
dtype: float64

## Step 6 – Write the predictions into the pile

Predictions of the tuned logistic regression for the **test set**: the real label, the prediction, and the **probability** of finding a home, with the model name and time. Saved as the Iceberg table **`gold.found_home_predictions`**.

In [10]:
predictions = (logreg_model.transform(test)
               .withColumn("prob_found_home", F.round(vector_to_array("probability")[1], 4))
               .withColumn("model", F.lit("logistic_regression_cv5"))
               .withColumn("predicted_at", F.current_timestamp())
               .select("id", "name", "species", "n_applications", "days_in_shelter",
                       F.col("label").cast("int").alias("found_home"),
                       F.col("prediction").cast("int").alias("predicted_found_home"),
                       "prob_found_home", "model", "predicted_at"))

pred_table = write_iceberg(predictions, "gold.found_home_predictions", mode="overwrite")
print("Rows in gold.found_home_predictions:", len(pred_table.scan().to_arrow()))

Rows in gold.found_home_predictions: 593


In [11]:
# Read back from the pile: animals the model thinks are LEAST likely to find a home (→ promote them!)
read_iceberg(spark, "gold.found_home_predictions") \
    .filter("found_home = 0").orderBy("prob_found_home") \
    .select("id", "name", "species", "n_applications", "days_in_shelter", "prob_found_home").show(10)

+----+------+----------+--------------+---------------+---------------+
|  id|  name|   species|n_applications|days_in_shelter|prob_found_home|
+----+------+----------+--------------+---------------+---------------+
| 414| Molly|       dog|             0|            181|         0.1418|
| 746|  Coco|       dog|             0|            128|          0.147|
|2446|  Jack|       dog|             0|            107|         0.1513|
|1818|  Finn|       dog|             0|             68|         0.1522|
|2677|Ginger|       dog|             0|            390|          0.154|
| 781| Bruno|guinea pig|             0|             92|          0.155|
|  52|Cooper|       dog|             0|            149|          0.156|
|1346|Gracie|guinea pig|             0|             48|         0.1562|
|1182| Teddy|       dog|             0|            434|         0.1565|
|1912| Buddy|       dog|             0|             50|         0.1565|
+----+------+----------+--------------+---------------+---------